In [305]:
import os
import pandas as pd
import matplotlib as plt
import seaborn as sns


In [306]:
df_avito = pd.read_csv("Bronze/apparts_vente.csv")

In [307]:
df_avito = df_avito.drop(
    columns=[
        
        'Meublé', 'Chauffage', 'Concierge', 'category', 'Sécurité',
        'Terrasse', 'Cuisine équipée', 'Balcon', 'Parking',
        'Climatisation', 'Ascenseur','Frais de syndic / mois', 'Disponibilité', 'Standing',
        'Surface totale', 'Âge du bien', 'seller_name', 'seller_phone', 'image_count', 
        'Salons', 'Étage', "Type d'appartement", 'seller_type', 'image_url','price_value'
    ]
)

In [308]:
df_avito = df_avito.rename(columns={
    'Surface habitable': 'surface_m2',
    'list_time': 'list_date',
    'Chambres': 'rooms',
    'Condition': 'condition',
    'Salle de bain': 'bathrooms',
})

In [309]:
def to_number(col):
    return pd.to_numeric(col.astype(str).str.replace(r"\D", "", regex=True), errors="coerce")

df_avito["price"] = to_number(df_avito["price"])
df_avito["surface_m2"] = to_number(df_avito["surface_m2"])
df_avito["list_date"] = pd.to_datetime(df_avito["list_date"], errors="coerce", utc=True).dt.tz_localize(None)
df_avito["area"] = df_avito["area"].astype("object")

In [310]:
df_avito.info()

<class 'pandas.DataFrame'>
RangeIndex: 41571 entries, 0 to 41570
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   description  41570 non-null  str           
 1   price        32898 non-null  float64       
 2   title        41571 non-null  str           
 3   area         0 non-null      object        
 4   surface_m2   40779 non-null  float64       
 5   list_date    41571 non-null  datetime64[us]
 6   city         41571 non-null  str           
 7   rooms        40997 non-null  float64       
 8   condition    31721 non-null  str           
 9   bathrooms    40041 non-null  float64       
 10  url          41571 non-null  str           
dtypes: datetime64[us](1), float64(4), object(1), str(5)
memory usage: 3.5+ MB


In [311]:
df_mubwab = pd.read_csv('Bronze/mubawab_listings.csv')

In [312]:
import re
from urllib.parse import unquote

# === 1. Extract dates from image_names ===
def extract_date_from_images(img_names):
    if pd.isna(img_names):
        return None
    decoded = unquote(str(img_names))
    
    patterns = [
        r'WhatsApp.*?(\d{4})-(\d{2})-(\d{2})',
        r'IMG-(\d{4})(\d{2})(\d{2})',
        r'(\d{4})(\d{2})(\d{2})_\d{6}',
        r'Generated.*?(\d{4}).*?(\d{2}).*?(\d{2})',
        r'(\d{4})(\d{2})(\d{2})',
    ]
    
    for pat in patterns:
        m = re.search(pat, decoded, re.IGNORECASE)
        if m:
            try:
                groups = m.groups()
                if len(groups) == 3:
                    y, mth, d = map(int, groups)
                    if 2015 <= y <= 2027 and 1 <= mth <= 12 and 1 <= d <= 31:
                        return f"{y:04d}-{mth:02d}-{d:02d}"
            except:
                continue
    return None

df_mubwab['extracted_date'] = df_mubwab['image_names'].apply(extract_date_from_images)
df_mubwab['listing_date'] = pd.to_datetime(df_mubwab['extracted_date'], errors='coerce')

# === 2. Fill missing dates using ad_id ===
df_mubwab['ad_id_num'] = pd.to_numeric(df_mubwab['ad_id'], errors='coerce')
df_mubwab = df_mubwab.sort_values('ad_id_num').reset_index(drop=True)

df_mubwab['listing_date_filled'] = df_mubwab['listing_date'].copy()
df_mubwab['listing_date_filled'] = df_mubwab['listing_date_filled'].interpolate(
    method='linear', 
    limit_direction='both'
)

# Final date
df_mubwab['listing_date'] = df_mubwab['listing_date'].combine_first(df_mubwab['listing_date_filled'])

# === NEW: Boolean column ===
df_mubwab['is_extracted'] = df_mubwab['listing_date'].notna()



In [313]:
split = df_mubwab['location'].str.split(r'\s*,+\s*', n=1, expand=True, regex=True)

df_mubwab['area'] = split[0].str.strip()
df_mubwab['city'] = split[1].str.strip()

In [314]:
df_mubwab = df_mubwab.drop(
    columns=[
        'location', 'ad_id', 'listing_type', 'orientation', 'floor_type',
        'garden_m2', 'garage_spaces', 'features_text', 'phone',
        'agency_name', 'agency_url', 'image_names', 'page', 'extracted_date',
        'listing_date', 'ad_id_num', 'is_extracted', 'bedrooms'
        
    ]
)

In [315]:
df_mubwab = df_mubwab.rename(columns={
    'price_DH': 'price',
    'listing_date_filled': 'list_date',
})

In [316]:

df_mubwab["price"] = to_number(df_mubwab["price"])

In [317]:
df_mubwab.info()

<class 'pandas.DataFrame'>
RangeIndex: 16658 entries, 0 to 16657
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   title        16658 non-null  str           
 1   price        15103 non-null  float64       
 2   surface_m2   16096 non-null  float64       
 3   rooms        14130 non-null  float64       
 4   bathrooms    6015 non-null   float64       
 5   condition    5922 non-null   str           
 6   description  7458 non-null   str           
 7   url          16658 non-null  str           
 8   list_date    16658 non-null  datetime64[us]
 9   area         16658 non-null  str           
 10  city         16539 non-null  str           
dtypes: datetime64[us](1), float64(4), str(6)
memory usage: 1.4 MB


In [318]:
df_mubwab.sample(10)

,title,price,surface_m2,rooms,bathrooms,condition,description,url,list_date,area,city
7497,RAFAA SQUARE – Une bouffée de modernité pour Salé,13500.0,93.0,1.0,NaN,NaN,NaN,https://www.mubawab.ma/fr/a/8223945/rafaa-squa...,2025-09-12 00:00:00.000000,Roustal,Salé
5088,Superbe appartement à vendre à Anfa. 5 pièces ...,1200000.0,97.0,5.0,NaN,NaN,NaN,https://www.mubawab.ma/fr/a/8174108/superbe-ap...,2025-04-13 18:00:00.000000,Anfa,Casablanca
11320,Bel appartement 85m2 à Val Fleury place de par...,1560000.0,85.0,4.0,NaN,Projet neuf,CHRAIBI IMMOBILIER – VAL FLEURY\r Résidence de...,https://www.mubawab.ma/fr/pa/8279848/bel-appar...,2025-12-11 04:08:16.551724,Val Fleury,Casablanca
7053,Appartement à l'achat à Hay Oued Dahab. 2 pièc...,34000000.0,84.0,2.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/8216390/appartemen...,2025-06-09 00:00:00.000000,Hay Oued Dahab,Salé
10306,Superbe appartement à vendre à Marjane. 9 gran...,760000.0,120.0,9.0,NaN,NaN,NaN,https://www.mubawab.ma/fr/a/8265241/superbe-ap...,2025-06-30 10:17:08.571428,Marjane,Meknes
5077,À vendre – 2 appartements au centre de Marrakech,1254000.0,57.0,2.0,1.0,Bon état / habitable,Idéal pour investissement locatif ou habitatio...,https://www.mubawab.ma/fr/a/8173704/%C3%A0-ven...,2025-07-03 12:00:00.000000,Guéliz,Marrakech
809,Ocean 3 - Appartements à vendre,375000.0,NaN,NaN,NaN,NaN,NaN,https://www.mubawab.ma/fr/p/4601/ocean-3-appar...,2024-08-26 00:00:00.000000,Ennajd,El Jadida
2169,Grand duplex 4 chambres majorelle,2800000.0,306.0,1.0,NaN,Bon état / habitable,MARRAKECH ️ À 2 minutes à pied de Majorelle e...,https://www.mubawab.ma/fr/a/7983130/grand-dupl...,2024-08-21 00:00:00.000000,Majorelle,Marrakech
15303,Appartement à vendre à Aïn Sebaâ. 2 chambres a...,1196000.0,104.0,NaN,NaN,NaN,La Résidences SAINT MAURICE propose des appart...,https://www.mubawab.ma/fr/pa/8329748/apparteme...,2026-04-07 00:00:00.000000,Aïn Sebaâ,Casablanca
9931,"Appartement à vendre 165 m², 3 chambres Les Hô...",1750000.0,165.0,NaN,NaN,NaN,Bel appartement de 136 m² avec balcon et garag...,https://www.mubawab.ma/fr/a/8262370/appartemen...,2025-02-07 00:00:00.000000,Les Hôpitaux,Casablanca


In [319]:
df_avito["listing_type"] = "sale"
df_mubwab["listing_type"] = "sale"

df_merged = pd.concat([df_avito, df_mubwab])

In [320]:
df_merged.to_csv("Silver/sales_clean.csv")